# Kredi Onay Tahmini — Lojistik Regresyon ve Sınıflandırma Modelleri

**Proje Konusu:** Lojistik Regresyon ile Kredi Onay Tahmini Sınıflandırma Modeli

**Amaç:** Bankalara kredi başvurusunda bulunan müşterilerin demografik ve finansal verilerini kullanarak
kredinin onaylanıp onaylanmayacağını tahmin etmek (ikili sınıflandırma).

**Veri seti:** [Loan Approval Prediction Dataset](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset)
(Kaggle) — 4269 gözlem, 13 değişken.

Bu defter, İstatistiksel Yazılımlar II dersi final ödevi kapsamında hazırlanmıştır. Lojistik regresyondan
başlayarak 9 farklı makine öğrenmesi sınıflandırıcısı (+ bonus XGBoost) GridSearchCV ile optimize edilmiş
ve performansları karşılaştırılmıştır. Sonuçların ve yorumların tam metni `README.md` dosyasındadır.

## Bölüm 1 — Verinin Yüklenmesi ve İlk İnceleme

In [ ]:
import pandas as pd
import numpy as np

# Veri setini okuma
data = pd.read_csv("loan_approval_dataset.csv")

# Sütun isimlerinde olası boşlukları (space) temizleme
data.columns = data.columns.str.strip()

print("--- Veri Setinin İlk 5 Gözlemi ---")
display(data.head())

print("\n--- Veri Seti Bilgisi (Info) ---")
data.info()

print("\n--- Eksik Gözlem Sayısı ---")
print(data.isnull().any().sum())

print("\n--- Betimsel İstatistikler ---")
import warnings
warnings.filterwarnings('ignore')  # gereksiz uyarilari silmek icin
display(data.describe().T)

## Bölüm 2 — Veri Ön İşleme ve Kategorik Değişkenleri Sayısallaştırma

In [ ]:
# 1. Gereksiz değişkenin silinmesi
df = data.drop("loan_id", axis=1).copy()

# 2. Kategorik (Object) değişkenlerin sayısallaştırılması (Binary Encoding)
df['education'] = df['education'].str.strip().map({'Graduate': 1, 'Not Graduate': 0})
df['self_employed'] = df['self_employed'].str.strip().map({'Yes': 1, 'No': 0})
df['loan_status'] = df['loan_status'].str.strip().map({'Approved': 1, 'Rejected': 0})

print("--- Sayısallaştırma İşlemi Sonrası İlk 5 Gözlem ---")
display(df.head())

## Korelasyon Matrisi (Heatmap)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5, annot_kws={"size": 10})
plt.title('Değişkenler Arası Korelasyon Matrisi (Heatmap)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Bölüm 3 — Keşifsel Veri Analizi (EDA) ve Görselleştirmeler

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

# 1. GRAFİK: Kredi Onay Durumu Dağılımı (Sınıf Dengesi Kontrolü)
plt.figure(figsize=(8, 6))
ax1 = sns.countplot(x="loan_status", data=df, palette="viridis")
plt.title("Kredi Onay Durumu Dağılımı\n(0: Reddedildi, 1: Onaylandı)", fontsize=14, fontweight='bold')
plt.xlabel("Kredi Durumu (loan_status)", fontsize=12)
plt.ylabel("Müşteri Sayısı", fontsize=12)

for p in ax1.patches:
    ax1.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=12, color='black', xytext=(0, 5), textcoords='offset points')
plt.show()

In [ ]:
# 2. GRAFİK: Kredi Skoru (CIBIL Score) ve Kredi Onayı İlişkisi
plt.figure(figsize=(10, 6))
sns.boxplot(x="loan_status", y="cibil_score", data=df, palette="Set2")
plt.title("Kredi Onayına Göre Kredi Skoru (CIBIL) Dağılımı", fontsize=14, fontweight='bold')
plt.xlabel("Kredi Durumu (0: Red, 1: Onay)", fontsize=12)
plt.ylabel("CIBIL Skoru", fontsize=12)
plt.show()

In [ ]:
# 3. GRAFİK: Yıllık Gelir ve Talep Edilen Kredi Miktarı İlişkisi
plt.figure(figsize=(12, 7))
sns.scatterplot(x="income_annum", y="loan_amount", hue="loan_status", data=df, palette="coolwarm", alpha=0.7)
plt.title("Yıllık Gelir vs. Talep Edilen Kredi (Onay Durumuna Göre)", fontsize=14, fontweight='bold')
plt.xlabel("Yıllık Gelir (Income Annum)", fontsize=12)
plt.ylabel("Talep Edilen Kredi Miktarı (Loan Amount)", fontsize=12)
plt.legend(title='Kredi Durumu (0: Red, 1: Onay)', loc='upper left')
plt.show()

## Bölüm 4 — Veri Ön İşleme ve Ölçeklendirme (Train/Test Split)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. X ve y ayrımı
y = df["loan_status"]
x = df.drop("loan_status", axis=1)

# 2. Train/Test Split (Eğitim ve Test setlerine ayırma)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

# 3. StandardScaler (Lojistik, MLP ve SVM için ŞARTTIR)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Veri %80 Eğitim, %20 Test olarak ayrıldı ve başarıyla ölçeklendirildi (Scaled).")

## Lojistik Regresyon

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as mt
import seaborn as sns
import pandas as pd

print("--- STATSMODELS İLE LOGIT MODELİ ---")
X_train_sm = sm.add_constant(X_train_scaled)
log_model_sm = sm.Logit(y_train, X_train_sm).fit()
print(log_model_sm.summary())

print("\n--- SKLEARN İLE LOJİSTİK REGRESYON MODELİ ---")
log_model = LogisticRegression(solver="liblinear", C=1e8).fit(X_train_scaled, y_train.values.ravel())

preds = log_model.predict(X_test_scaled)

acc = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds)
cm = confusion_matrix(y_test, preds)

print("Karmaşıklık Matrisi (Confusion Matrix):\n", cm)
print(f"Accuracy Skoru: {acc:.4f}")
print(f"F1 Skoru: {f1:.4f}")

print("\n--- ÇAPRAZ DOĞRULAMA (CROSS-VALIDATION) ---")
cv_scores = cross_val_score(log_model, X_test_scaled, y_test.values.ravel(), cv=10)
print("10 Katlı CV Ortalama Skoru:", cv_scores.mean())

In [ ]:
fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("Karmaşıklık Matrisi (Heatmap)")
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

auc_degeri = roc_auc_score(y_test, preds)
fpr, tpr, treshold_val = roc_curve(y_test, log_model.predict_proba(X_test_scaled)[:, 1])

axes[1].plot(fpr, tpr, color="darkorange", label=f"AUC = {auc_degeri:.4f}")
axes[1].plot([0, 1], [0, 1], "r--")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Eğrisi")
axes[1].legend(loc="lower right")

katsayilar = pd.Series(log_model.coef_[0], index=x.columns).sort_values()
katsayilar.plot(kind="barh", ax=axes[2], color="steelblue")
axes[2].set_title("Lojistik Regresyon Katsayı Etkileri")
axes[2].set_xlabel("Katsayı Büyüklüğü")

mt.tight_layout()
mt.show()

## Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import cross_val_score

naive_model = GaussianNB().fit(X_train_scaled, y_train.values.ravel())

preds_naive = naive_model.predict(X_test_scaled)
probs_naive = naive_model.predict_log_proba(X_test_scaled)

acc_naive = accuracy_score(y_test, preds_naive)
f1_naive = f1_score(y_test, preds_naive)
cm_naive = confusion_matrix(y_test, preds_naive)

print("Karmaşıklık Matrisi (Confusion Matrix):\n", cm_naive)
print(f"Naive Bayes Accuracy (Doğruluk) Skoru: {acc_naive:.4f}")
print(f"Naive Bayes F1-Skoru: {f1_naive:.4f}")

print("\n--- ÇAPRAZ DOĞRULAMA (CROSS-VALIDATION) ---")
cv_scores_naive = cross_val_score(naive_model, X_train_scaled, y_train.values.ravel(), cv=10)
print(f"10 Katlı CV Ortalama Skoru: {cv_scores_naive.mean():.4f}")

In [ ]:
import matplotlib.pyplot as mt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve

fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_naive, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("Naive Bayes Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

nb_probs = naive_model.predict_proba(X_test_scaled)[:, 1]
auc_nb = roc_auc_score(y_test, nb_probs)
fpr_nb, tpr_nb, thresholds_nb = roc_curve(y_test, nb_probs)

axes[1].plot(fpr_nb, tpr_nb, color="darkorange", lw=2, label=f"AUC = {auc_nb:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Naive Bayes ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

sns.histplot(nb_probs, kde=True, ax=axes[2], color="purple", bins=20)
axes[2].set_title("Tahmin Olasılıkları Dağılımı", fontweight='bold')
axes[2].set_xlabel("Onaylanma İhtimali (Probability)")
axes[2].set_ylabel("Frekans")

mt.tight_layout()
mt.show()

## Doğrusal Olmayan Sınıflandırma Modelleri
### K-En Yakın Komşu (K-NN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
import numpy as np

knn_model = KNeighborsClassifier().fit(X_train_scaled, y_train.values.ravel())
preds_knn = knn_model.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, preds_knn)
f1_knn = f1_score(y_test, preds_knn)
print(f"Varsayılan KNN Accuracy: {acc_knn:.4f}")
print(f"Varsayılan KNN F1-Skoru: {f1_knn:.4f}")

print("\n--- K-NN OPTİMİZASYONU (GRID SEARCH) ---")
knn_params = {"n_neighbors": np.arange(1, 40)}
knn_mod = KNeighborsClassifier()
knn_cv = GridSearchCV(knn_mod, knn_params, cv=10).fit(X_train_scaled, y_train.values.ravel())
best_k = knn_cv.best_params_['n_neighbors']
print(f"Grid Search ile Bulunan En İyi Komşu Sayısı (K): {best_k}")

print("\n--- NİHAİ (TUNED) K-NN MODELİ ---")
knn_model_best = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_scaled, y_train.values.ravel())
preds_knn_best = knn_model_best.predict(X_test_scaled)
acc_knn_best = accuracy_score(y_test, preds_knn_best)
f1_knn_best = f1_score(y_test, preds_knn_best)
cm_knn_best = confusion_matrix(y_test, preds_knn_best)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_knn_best)
print(f"Nihai KNN Accuracy: {acc_knn_best:.4f}")
print(f"Nihai KNN F1-Skoru: {f1_knn_best:.4f}")

In [ ]:
import matplotlib.pyplot as mt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve

fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_knn_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("Nihai K-NN Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

knn_probs = knn_model_best.predict_proba(X_test_scaled)[:, 1]
auc_knn = roc_auc_score(y_test, knn_probs)
fpr_k, tpr_k, thresholds_k = roc_curve(y_test, knn_probs)

axes[1].plot(fpr_k, tpr_k, color="darkorange", lw=2, label=f"AUC = {auc_knn:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Nihai K-NN ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

mean_scores = knn_cv.cv_results_['mean_test_score']
k_values = knn_params['n_neighbors']

axes[2].plot(k_values, mean_scores, marker='o', linestyle='solid', color='steelblue', markersize=5)
axes[2].axvline(x=best_k, color='red', linestyle='--', lw=2, label=f'Optimum K: {best_k}')
axes[2].set_title("K-NN Optimizasyon (Grid Search) Eğrisi", fontweight='bold')
axes[2].set_xlabel("Komşu Sayısı (K)")
axes[2].set_ylabel("Ortalama Doğruluk (Accuracy)")
axes[2].legend()

mt.tight_layout()
mt.show()

## Destek Vektör Makineleri (SVM)
### Doğrusal SVM (Linear SVC)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
import numpy as np

svm_model = SVC(kernel="linear").fit(X_train_scaled, y_train.values.ravel())
svm_preds = svm_model.predict(X_test_scaled)
acc_svm = accuracy_score(y_test, svm_preds)
f1_svm = f1_score(y_test, svm_preds)
print(f"Varsayılan Doğrusal SVM Accuracy: {acc_svm:.4f}")
print(f"Varsayılan Doğrusal SVM F1-Skoru: {f1_svm:.4f}")

print("\n--- DOĞRUSAL SVM OPTİMİZASYONU (GRID SEARCH) ---")
svm_parms = {"C": np.arange(1, 15)}
svm_mod = SVC(kernel="linear")
svm_cv = GridSearchCV(svm_mod, svm_parms, cv=10, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())
best_c_linear = svm_cv.best_params_['C']
print(f"\nGrid Search ile Bulunan Optimum 'C' Değeri: {best_c_linear}")

print("\n--- NİHAİ (TUNED) DOĞRUSAL SVM MODELİ ---")
svm_best = SVC(kernel="linear", C=best_c_linear).fit(X_train_scaled, y_train.values.ravel())
svm_best_preds = svm_best.predict(X_test_scaled)
acc_svm_best = accuracy_score(y_test, svm_best_preds)
f1_svm_best = f1_score(y_test, svm_best_preds)
cm_svm_best = confusion_matrix(y_test, svm_best_preds)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_svm_best)
print(f"Nihai Doğrusal SVM Accuracy: {acc_svm_best:.4f}")
print(f"Nihai Doğrusal SVM F1-Skoru: {f1_svm_best:.4f}")

In [ ]:
import matplotlib.pyplot as mt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np

fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_svm_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("Doğrusal SVM Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

svm_scores = svm_best.decision_function(X_test_scaled)
auc_svm = roc_auc_score(y_test, svm_scores)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_scores)

axes[1].plot(fpr_svm, tpr_svm, color="darkorange", lw=2, label=f"AUC = {auc_svm:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Doğrusal SVM ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

c_values = np.arange(1, 15)
mean_scores_svm = svm_cv.cv_results_['mean_test_score']

axes[2].plot(c_values, mean_scores_svm, marker='o', color='steelblue', linestyle='solid', markersize=6)
axes[2].axvline(x=best_c_linear, color='red', linestyle='--', lw=2, label=f'Optimum C: {best_c_linear}')
axes[2].set_title("Doğrusal SVM: C Parametresi Optimizasyon Eğrisi", fontweight='bold')
axes[2].set_xlabel("C Değeri (Ceza Parametresi)")
axes[2].set_ylabel("Ortalama Doğruluk (Accuracy)")
axes[2].legend()

mt.tight_layout()
mt.show()

### Doğrusal Olmayan SVM (RBF Kernel SVC)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

svm_model_rbf = SVC(kernel="rbf").fit(X_train_scaled, y_train.values.ravel())
svm_preds_rbf = svm_model_rbf.predict(X_test_scaled)
acc_svm_rbf = accuracy_score(y_test, svm_preds_rbf)
f1_svm_rbf = f1_score(y_test, svm_preds_rbf)
print(f"Varsayılan RBF SVM Accuracy: {acc_svm_rbf:.4f}")
print(f"Varsayılan RBF SVM F1-Skoru: {f1_svm_rbf:.4f}")

print("\n--- RBF SVM OPTİMİZASYONU (GRID SEARCH) ---")
svm_parms_rbf = {"C": [1, 10, 100, 1000], "gamma": [1, 0.1, 0.01, 0.001]}
svm_mod_rbf = SVC(kernel="rbf")
svm_cv_rbf = GridSearchCV(svm_mod_rbf, svm_parms_rbf, cv=10, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())
best_c_rbf = svm_cv_rbf.best_params_['C']
best_gamma_rbf = svm_cv_rbf.best_params_['gamma']
print(f"\nGrid Search ile Bulunan Optimum 'C': {best_c_rbf}")
print(f"Grid Search ile Bulunan Optimum 'Gamma': {best_gamma_rbf}")

print("\n--- NİHAİ (TUNED) RBF SVM MODELİ ---")
svm_best_rbf = SVC(kernel="rbf", C=best_c_rbf, gamma=best_gamma_rbf).fit(X_train_scaled, y_train.values.ravel())
svm_best_preds_rbf = svm_best_rbf.predict(X_test_scaled)
acc_svm_best_rbf = accuracy_score(y_test, svm_best_preds_rbf)
f1_svm_best_rbf = f1_score(y_test, svm_best_preds_rbf)
cm_svm_best_rbf = confusion_matrix(y_test, svm_best_preds_rbf)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_svm_best_rbf)
print(f"Nihai RBF SVM Accuracy: {acc_svm_best_rbf:.4f}")
print(f"Nihai RBF SVM F1-Skoru: {f1_svm_best_rbf:.4f}")

In [ ]:
# SVM'e özel görsel: 2 boyutlu karar sınırı (Decision Boundary), PCA ile indirgenmiş
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import SVC

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)

svm_pca = SVC(kernel="rbf", C=best_c_rbf, gamma=best_gamma_rbf)
svm_pca.fit(X_train_pca, y_train.values.ravel())

x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
scatter = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train.values.ravel(), s=30, edgecolor='k', cmap='coolwarm')
plt.title('RBF SVM: Doğrusal Olmayan Karar Sınırı (Decision Boundary)', fontsize=14, fontweight='bold')
plt.xlabel('Verinin 1. Temel Bileşeni (PCA 1)')
plt.ylabel('Verinin 2. Temel Bileşeni (PCA 2)')
plt.legend(*scatter.legend_elements(), title="Gerçek Sınıf (0: Ret, 1: Onay)", loc="upper right")
plt.tight_layout()
plt.show()

## Yapay Sinir Ağları (MLP)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

mlp_model = MLPClassifier(max_iter=500, random_state=42).fit(X_train_scaled, y_train.values.ravel())
mlp_preds = mlp_model.predict(X_test_scaled)
acc_mlp = accuracy_score(y_test, mlp_preds)
f1_mlp = f1_score(y_test, mlp_preds)
print(f"Varsayılan MLP Accuracy: {acc_mlp:.4f}")
print(f"Varsayılan MLP F1-Skoru: {f1_mlp:.4f}")

print("\n--- MLP OPTİMİZASYONU (HIZLI GRID SEARCH) ---")
mlp_parms_fast = {
    "hidden_layer_sizes": [(20, 20), (30, 30)],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [0.0001, 0.01]
}
mlp_fast = MLPClassifier(max_iter=500, random_state=42)
mlp_cv_fast = GridSearchCV(mlp_fast, mlp_parms_fast, cv=3, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())

print("\nHızlı Grid Search ile Bulunan En İyi Parametreler:")
for param, value in mlp_cv_fast.best_params_.items():
    print(f" - {param}: {value}")

print("\n--- NİHAİ (TUNED) MLP MODELİ ---")
mlp_best_fast = MLPClassifier(**mlp_cv_fast.best_params_, max_iter=500, random_state=42).fit(X_train_scaled, y_train.values.ravel())
best_mlp_preds_fast = mlp_best_fast.predict(X_test_scaled)
acc_mlp_best = accuracy_score(y_test, best_mlp_preds_fast)
f1_mlp_best = f1_score(y_test, best_mlp_preds_fast)
cm_mlp_best = confusion_matrix(y_test, best_mlp_preds_fast)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_mlp_best)
print(f"Nihai MLP Accuracy: {acc_mlp_best:.4f}")
print(f"Nihai MLP F1-Skoru: {f1_mlp_best:.4f}")

In [ ]:
import matplotlib.pyplot as mt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve

fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_mlp_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("MLP Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

mlp_probs = mlp_best_fast.predict_proba(X_test_scaled)[:, 1]
auc_mlp = roc_auc_score(y_test, mlp_probs)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_probs)

axes[1].plot(fpr_mlp, tpr_mlp, color="darkorange", lw=2, label=f"AUC = {auc_mlp:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("MLP ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

axes[2].plot(mlp_best_fast.loss_curve_, color="purple", lw=2)
axes[2].set_title("MLP Öğrenme Eğrisi (Loss Curve)", fontweight='bold')
axes[2].set_xlabel("İterasyon Sayısı (Epoch)")
axes[2].set_ylabel("Kayıp (Loss)")
axes[2].grid(True, alpha=0.3)

mt.tight_layout()
mt.show()

## Karar Ağaçları (Decision Tree / CART)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

cart_model = DecisionTreeClassifier(random_state=42).fit(X_train_scaled, y_train.values.ravel())
cart_preds = cart_model.predict(X_test_scaled)
acc_cart = accuracy_score(y_test, cart_preds)
f1_cart = f1_score(y_test, cart_preds)
print(f"Varsayılan CART Accuracy: {acc_cart:.4f}")
print(f"Varsayılan CART F1-Skoru: {f1_cart:.4f}")

print("\n--- CART OPTİMİZASYONU (GRID SEARCH) ---")
cart_parms = {"max_depth": [3, 5, 7, 9], "min_samples_split": [2, 5, 10]}
cart = DecisionTreeClassifier(random_state=42)
cart_cv = GridSearchCV(cart, cart_parms, cv=10, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())

print("\nGrid Search ile Bulunan En İyi Parametreler:")
for param, value in cart_cv.best_params_.items():
    print(f" - {param}: {value}")

print("\n--- NİHAİ (TUNED) KARAR AĞACI MODELİ ---")
cart_best = DecisionTreeClassifier(**cart_cv.best_params_, random_state=42).fit(X_train_scaled, y_train.values.ravel())
cart_best_preds = cart_best.predict(X_test_scaled)
acc_cart_best = accuracy_score(y_test, cart_best_preds)
f1_cart_best = f1_score(y_test, cart_best_preds)
cm_cart_best = confusion_matrix(y_test, cart_best_preds)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_cart_best)
print(f"Nihai CART Accuracy: {acc_cart_best:.4f}")
print(f"Nihai CART F1-Skoru: {f1_cart_best:.4f}")

In [ ]:
fig, axes = mt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_cart_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("CART Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

derinlikler = range(1, 15)
train_scores = []
test_scores = []
for d in derinlikler:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train_scaled, y_train)
    train_scores.append(accuracy_score(y_train, dt.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test_scaled)))

axes[1].plot(derinlikler, train_scores, marker='o', label='Eğitim Seti', color='steelblue')
axes[1].plot(derinlikler, test_scores, marker='s', label='Test Seti', color='darkorange')
axes[1].axvline(x=9, color='red', linestyle='--', lw=2, label='Optimum Derinlik: 9')
axes[1].set_title("CART: Ağaç Derinliğine Göre Doğruluk", fontweight='bold')
axes[1].set_xlabel("Maksimum Derinlik (max_depth)")
axes[1].set_ylabel("Doğruluk (Accuracy)")
axes[1].legend()

mt.tight_layout()
mt.show()

## Topluluk Modelleri (Ensemble Learning)
### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
import pandas as pd
import matplotlib.pyplot as mt
import seaborn as sns

rf_model = RandomForestClassifier(random_state=42).fit(X_train_scaled, y_train.values.ravel())
rf_preds = rf_model.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, rf_preds)
f1_rf = f1_score(y_test, rf_preds)
print(f"Varsayılan RF Accuracy: {acc_rf:.4f}")
print(f"Varsayılan RF F1-Skoru: {f1_rf:.4f}")

print("\n--- RANDOM FOREST OPTİMİZASYONU (GRID SEARCH) ---")
rf_parms = {"n_estimators": [100, 200, 300], "max_depth": [3, 5, 7], "min_samples_split": [2, 5, 10]}
rf_mod = RandomForestClassifier(random_state=42)
rf_cv = GridSearchCV(rf_mod, rf_parms, cv=5, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())

print("\nGrid Search ile Bulunan En İyi Parametreler:")
for param, value in rf_cv.best_params_.items():
    print(f" - {param}: {value}")

print("\n--- NİHAİ (TUNED) RANDOM FOREST MODELİ ---")
rf_best = RandomForestClassifier(**rf_cv.best_params_, random_state=42).fit(X_train_scaled, y_train.values.ravel())
rf_best_preds = rf_best.predict(X_test_scaled)
acc_rf_best = accuracy_score(y_test, rf_best_preds)
f1_rf_best = f1_score(y_test, rf_best_preds)
cm_rf_best = confusion_matrix(y_test, rf_best_preds)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_rf_best)
print(f"Nihai RF Accuracy: {acc_rf_best:.4f}")
print(f"Nihai RF F1-Skoru: {f1_rf_best:.4f}")

In [ ]:
fig, axes = mt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_rf_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("Random Forest Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

from sklearn.metrics import roc_auc_score, roc_curve
rf_probs = rf_best.predict_proba(X_test_scaled)[:, 1]
auc_rf = roc_auc_score(y_test, rf_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

axes[1].plot(fpr_rf, tpr_rf, color="darkorange", lw=2, label=f"AUC = {auc_rf:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Random Forest ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

mt.tight_layout()
mt.show()

In [ ]:
# Değişken Önem Düzeyi (Feature Importance)
importances = rf_best.feature_importances_
features = X_train.columns

rf_imp_df = pd.DataFrame({'Değişken': features, 'Önem Düzeyi': importances})
rf_imp_df = rf_imp_df.sort_values(by='Önem Düzeyi', ascending=False)

mt.figure(figsize=(10, 6))
sns.barplot(x='Önem Düzeyi', y='Değişken', data=rf_imp_df, palette='magma')
mt.title("Random Forest: Değişken Önem Düzeyleri (Feature Importance)", fontweight='bold', fontsize=14)
mt.xlabel("Önem Düzeyi (Etki Skoru)", fontsize=12)
mt.ylabel("Değişkenler", fontsize=12)
mt.tight_layout()
mt.show()

### Gradient Boosting (GBM)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
import pandas as pd
import matplotlib.pyplot as mt
import seaborn as sns

gbm_model = GradientBoostingClassifier(random_state=42).fit(X_train_scaled, y_train.values.ravel())
gbm_preds = gbm_model.predict(X_test_scaled)
acc_gbm = accuracy_score(y_test, gbm_preds)
f1_gbm = f1_score(y_test, gbm_preds)
print(f"Varsayılan GBM Accuracy: {acc_gbm:.4f}")
print(f"Varsayılan GBM F1-Skoru: {f1_gbm:.4f}")

print("\n--- GBM OPTİMİZASYONU (GRID SEARCH) ---")
gbm_parms = {"n_estimators": [100, 200, 300], "learning_rate": [0.01, 0.1, 0.2], "max_depth": [3, 5, 7]}
gbm_mod = GradientBoostingClassifier(random_state=42)
gbm_cv = GridSearchCV(gbm_mod, gbm_parms, cv=5, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())

print("\nGrid Search ile Bulunan En İyi Parametreler:")
for param, value in gbm_cv.best_params_.items():
    print(f" - {param}: {value}")

print("\n--- NİHAİ (TUNED) GBM MODELİ ---")
gbm_best = GradientBoostingClassifier(**gbm_cv.best_params_, random_state=42).fit(X_train_scaled, y_train.values.ravel())
gbm_best_preds = gbm_best.predict(X_test_scaled)
acc_gbm_best = accuracy_score(y_test, gbm_best_preds)
f1_gbm_best = f1_score(y_test, gbm_best_preds)
cm_gbm_best = confusion_matrix(y_test, gbm_best_preds)

print("Nihai Karmaşıklık Matrisi (Confusion Matrix):\n", cm_gbm_best)
print(f"Nihai GBM Accuracy: {acc_gbm_best:.4f}")
print(f"Nihai GBM F1-Skoru: {f1_gbm_best:.4f}")

In [ ]:
fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_gbm_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("GBM Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

from sklearn.metrics import roc_auc_score, roc_curve
gbm_probs = gbm_best.predict_proba(X_test_scaled)[:, 1]
auc_gbm = roc_auc_score(y_test, gbm_probs)
fpr_gbm, tpr_gbm, _ = roc_curve(y_test, gbm_probs)

axes[1].plot(fpr_gbm, tpr_gbm, color="darkorange", lw=2, label=f"AUC = {auc_gbm:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("GBM ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

iterations = np.arange(1, gbm_best.n_estimators_ + 1)
axes[2].plot(iterations, gbm_best.train_score_, color="steelblue", lw=2, label="Eğitim Kaybı")
axes[2].set_title("GBM: Ardışık Öğrenme (Deviance) Eğrisi", fontweight='bold')
axes[2].set_xlabel("İterasyon Sayısı (Ağaç Sayısı)")
axes[2].set_ylabel("Kayıp (Deviance)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

mt.tight_layout()
mt.show()

In [ ]:
importances_gbm = gbm_best.feature_importances_
features_gbm = X_train.columns

gbm_imp_df = pd.DataFrame({'Değişken': features_gbm, 'Önem Düzeyi': importances_gbm})
gbm_imp_df = gbm_imp_df.sort_values(by='Önem Düzeyi', ascending=False)

mt.figure(figsize=(10, 6))
sns.barplot(x='Önem Düzeyi', y='Değişken', data=gbm_imp_df, palette='viridis')
mt.title("Gradient Boosting: Değişken Önem Düzeyleri (Feature Importance)", fontweight='bold', fontsize=14)
mt.xlabel("Önem Düzeyi (Etki Skoru)", fontsize=12)
mt.ylabel("Değişkenler", fontsize=12)
mt.tight_layout()
mt.show()

## Bonus Model — XGBoost (GBM ile Karşılaştırma)

In [ ]:
# Not: XGBoost, derste işlenen Gradient Boosting'in endüstri standardı
# haline gelmiş gelişmiş versiyonudur. GBM ile doğrudan karşılaştırmak
# amacıyla eklenmiştir.
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

xgb_model = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train_scaled, y_train.values.ravel())
xgb_preds = xgb_model.predict(X_test_scaled)
acc_xgb = accuracy_score(y_test, xgb_preds)
f1_xgb = f1_score(y_test, xgb_preds)
print(f"Varsayılan XGBoost Accuracy: {acc_xgb:.4f}")
print(f"Varsayılan XGBoost F1-Skoru: {f1_xgb:.4f}")

print("\n--- XGBOOST OPTİMİZASYONU (GRID SEARCH) ---")
xgb_parms = {"n_estimators": [100, 200, 300], "learning_rate": [0.01, 0.1, 0.2], "max_depth": [3, 5, 7]}
xgb_mod = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_cv = GridSearchCV(xgb_mod, xgb_parms, cv=5, n_jobs=-1, verbose=2).fit(X_train_scaled, y_train.values.ravel())

print("\nGrid Search ile Bulunan En İyi Parametreler:")
for param, value in xgb_cv.best_params_.items():
    print(f" - {param}: {value}")

print("\n--- NİHAİ (TUNED) XGBOOST MODELİ ---")
xgb_best = XGBClassifier(**xgb_cv.best_params_, random_state=42, eval_metric='logloss').fit(X_train_scaled, y_train.values.ravel())
xgb_best_preds = xgb_best.predict(X_test_scaled)
acc_xgb_best = accuracy_score(y_test, xgb_best_preds)
f1_xgb_best = f1_score(y_test, xgb_best_preds)
cm_xgb_best = confusion_matrix(y_test, xgb_best_preds)

print("Nihai Karmaşıklık Matrisi:\n", cm_xgb_best)
print(f"Nihai XGBoost Accuracy: {acc_xgb_best:.4f}")
print(f"Nihai XGBoost F1-Skoru: {f1_xgb_best:.4f}")

In [ ]:
import matplotlib.pyplot as mt
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

fig, axes = mt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_xgb_best, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False, annot_kws={"size": 12})
axes[0].set_title("XGBoost Karmaşıklık Matrisi", fontweight='bold')
axes[0].set_xlabel("Tahmin Edilen (Predicted)")
axes[0].set_ylabel("Gerçek (Actual)")

xgb_probs = xgb_best.predict_proba(X_test_scaled)[:, 1]
auc_xgb = roc_auc_score(y_test, xgb_probs)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_probs)

axes[1].plot(fpr_xgb, tpr_xgb, color="darkorange", lw=2, label=f"AUC = {auc_xgb:.4f}")
axes[1].plot([0, 1], [0, 1], "r--", lw=2)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("XGBoost ROC Eğrisi", fontweight='bold')
axes[1].legend(loc="lower right")

importances_xgb = xgb_best.feature_importances_
features_xgb = X_train.columns
xgb_imp_df = pd.DataFrame({'Değişken': features_xgb, 'Önem Düzeyi': importances_xgb})
xgb_imp_df = xgb_imp_df.sort_values(by='Önem Düzeyi', ascending=False)

sns.barplot(x='Önem Düzeyi', y='Değişken', data=xgb_imp_df, ax=axes[2], palette='rocket')
axes[2].set_title("XGBoost Değişken Önem Düzeyi", fontweight='bold')
axes[2].set_xlabel("Etki Skoru")
axes[2].set_ylabel("")

mt.tight_layout()
mt.show()

## Genel Model Karşılaştırması ve Final Sonucu

In [ ]:
import pandas as pd
import matplotlib.pyplot as mt
import seaborn as sns
from IPython.display import display

model_isimleri = [
    "Lojistik Regresyon", "Naive Bayes", "K-NN",
    "Doğrusal SVM", "RBF SVM", "Yapay Sinir Ağları",
    "Karar Ağaçları", "Random Forest", "Gradient Boosting"
]

# (Proje boyunca elde ettiğimiz en iyi skorlar)
accuracy_skorlari = [0.9052, 0.9368, 0.9063, 0.9169, 0.9520, 0.9660, 0.9696, 0.9731, 0.9848]
f1_skorlari = [0.9248, 0.9490, 0.9248, 0.9331, 0.9618, 0.9730, 0.9760, 0.9786, 0.9879]

sonuclar_df = pd.DataFrame({
    'Model': model_isimleri,
    'Accuracy': accuracy_skorlari,
    'F1_Skoru': f1_skorlari
})

sonuclar_df = sonuclar_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

print("--- TÜM MODELLERİN FİNAL PERFORMANS TABLOSU ---")
display(sonuclar_df)

fig, axes = mt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x='Accuracy', y='Model', data=sonuclar_df, ax=axes[0], palette='Blues_r')
axes[0].set_title('Modellerin Doğruluk (Accuracy) Karşılaştırması', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Accuracy Skoru', fontsize=12)
axes[0].set_ylabel('')
axes[0].set_xlim(0.85, 1.0)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.4f}", (p.get_width() + 0.002, p.get_y() + p.get_height() / 2),
                      ha='left', va='center', fontsize=11, fontweight='bold')

sns.barplot(x='F1_Skoru', y='Model', data=sonuclar_df, ax=axes[1], palette='Greens_r')
axes[1].set_title('Modellerin F1-Skoru Karşılaştırması', fontweight='bold', fontsize=14)
axes[1].set_xlabel('F1 Skoru', fontsize=12)
axes[1].set_ylabel('')
axes[1].set_xlim(0.85, 1.0)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_width():.4f}", (p.get_width() + 0.002, p.get_y() + p.get_height() / 2),
                      ha='left', va='center', fontsize=11, fontweight='bold')

mt.tight_layout()
mt.show()

## Sonuç

**Şampiyon model:** Gradient Boosting (GBM) — %98.48 Accuracy, 0.9879 F1-Skoru.

Detaylı yorumlar, model karşılaştırma tablosu ve iş çıkarımları için bkz. `README.md`.